<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Interrupted time series

**[Interrupted time series](https://learning.nextia-ai.com/courses/experiments/m05/interrupted-time-series/)** · Experiments and Causal Impact: Did Our Change Work? · Module 5, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the recommendation block's effect from one series of daily orders: model the trend, the weekday, the holidays and the sale before the launch, and read the step on the launch day.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | trend and season, from the [forecasting course](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/); a regression with yes-or-no terms. |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/interrupted-time-series/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M05-L04-interrupted-time-series/interrupted-time-series-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "launch/daily.csv": "05beaad911189494268554f480dcb7102a122486520de7f0c948e4661f1b3311",
    "launch/truth.json": "564f8f216dba98154249edf9e78d4c1a9be23c8097559368e44a690d3af19de5"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Nine days, one step

A made-up series: the trend adds 2 orders a day, and a change on day 6 adds a step. One regression finds both: a `day` term for the trend and a yes-or-no `post` term for the step.

In [ ]:
tiny = pd.DataFrame({"day": range(1, 10), "orders": [102, 104, 106, 108, 110, 122, 124, 126, 128]})
tiny["post"] = (tiny.day >= 6).astype(int)
fit = smf.ols("orders ~ day + post", data=tiny).fit()
print(f"trend {fit.params['day']:+.1f} a day, step {fit.params['post']:+.1f}")

> **Check.** You should see `trend +2.0 a day, step +10.0`.

## 2. The block's launch: the naive answer

The block went live for everyone on Monday 6 October 2025. Thanksgiving (13 October) and an autumn sale (9 to 19 October) fell in the same weeks. All the data is synthetic.

> **Predict.** The naive change of sales was +8.3% (a practice problem we made on purpose). Will the naive change of **orders** be higher or lower?

In [ ]:
daily = pd.read_csv("data/launch/daily.csv", parse_dates=["date"])
launch = pd.Timestamp("2025-10-06")
before = daily[(daily.date >= launch - pd.Timedelta(weeks=4)) & (daily.date < launch)].orders.mean()
after = daily[(daily.date >= launch) & (daily.date < launch + pd.Timedelta(weeks=4))].orders.mean()
print(f"naive change of orders: {after / before - 1:+.1%}")
print(f"true effect (simulator): {truth('launch')['block_effect_on_orders']:+.1%}")

> **Check.** You should see `naive change of orders: +10.2%` and `true effect (simulator): +2.0%`.

## 3. Trend and weekday only

The same model as `xp its --simple`: 8 weeks before and 4 weeks after, the log of daily orders, a trend `t`, the weekday and the step `post`. The errors are robust to 7 days of autocorrelation (`HAC`): neighbouring days are alike.

In [ ]:
daily["t"] = (daily.date - daily.date.min()).dt.days
daily["post"] = (daily.date >= launch).astype(int)
daily["log_orders"] = np.log(daily.orders)
w = daily[(daily.date >= launch - pd.Timedelta(weeks=8)) & (daily.date < launch + pd.Timedelta(weeks=4))].copy()
simple = smf.ols("log_orders ~ post + t + C(weekday)", data=w).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
print(f"{len(w)} days; effect {np.expm1(simple.params['post']):+.1%}")

> **Check.** You should see `84 days; effect +11.3%`.

Thanksgiving and the sale fall after the launch, and this model knows nothing about them. The step takes their orders.

## 4. Your turn: add the calendar

The data has two yes-or-no columns: `holiday_window` (the shopping days around a holiday) and `sale` (9 to 19 October).

> **Your turn.** Fit the model with `holiday_window` and `sale` added, with the same errors. Put the effect, `np.expm1` of the `post` coefficient rounded to 4 decimals, in `its_full`.

In [ ]:
m = smf.ols("log_orders ~ post + t + C(weekday) + holiday_window + sale", data=w).fit(
    cov_type="HAC", cov_kwds={"maxlags": 7})
its_full = round(float(np.expm1(m.params["post"])), 4)
print(its_full)

In [ ]:
expect('its_full', its_full, 0.0349)

The estimate falls from +11.3% to about +3.5%, against a true +2.0%. Over 50 seeds, the course measured this model's interval containing the truth in 84% of the runs; the simple model's in 14%.

## 5. The model without its step

Draw the real orders and the model's counterfactual: the fitted model with `post` set to 0. The gap after the launch is the effect.

In [ ]:
import matplotlib.pyplot as plt

full = smf.ols("log_orders ~ post + t + C(weekday) + holiday_window + sale", data=w).fit()
cf = np.exp(full.predict(w.assign(post=0)))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(w.date, w.orders, label="orders")
ax.plot(w.date, cf, "--", label="model without the step")
ax.axvline(launch, color="grey", linestyle=":")
ax.set_ylabel("orders a day")
ax.legend()
plt.show()

The dashed line follows Thanksgiving and the sale, so the bump after the launch is not given to the block.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m05/interrupted-time-series/) compares four models with the truth, puts the module's four methods side by side, and has the knowledge checks that count toward your certificate. In the project: `sim run --scenario launch`, then `xp its` and `xp its --simple`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Regression adjustment](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m05/interrupted-time-series/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.